In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pylcp as pylcp 
import scipy.constants as cts

In [ ]:
mass_lab = 173*cts.value('atomic mass constant') # YB-173 mass in kg



klab = 2*np.pi*25068.2222
# Lab wavevector (without 2pi) in cm^{-1}   # Lifetime of 6P_{3/2} state (in seconds)
gammalab = (1.93*1e8)
Blab = -67*1e-3#(v0*cts.hbar*(klab*100))/cts.value('Bohr magneton')
b_0L =  48*1e-3
# (v0*cts.hbar*(klab*100))/cts.value('Bohr magneton')/2
# T # About 15 G/cm is a typical gradient for Rb
L = 16
x0 = 1/klab  # cm
t0 = 1/gammalab  # s
gamma = 1
k = 1
#100*(mass_lab*(v0**2))/(0.5*cts.hbar*(klab*100)*gammalab) +5
print(Blab, L,b_0L)

In [ ]:
H_g_D2, mu_q_g_D2 = pylcp.hamiltonians.hyperfine_coupled(
3/2, 9/2, 1.035,-0.2592,
    Ahfs = 0, Bhfs=0, Chfs=0,
    muB=1)# ground state 1s0#173-gI factor -0.2592
H_e_D2, mu_q_e_D2 = pylcp.hamiltonians.hyperfine_coupled(
5/2, 9/2, 0.05,-0.2592,
    Ahfs=59.521*1e6/gammalab,Bhfs = 601.87*1e6/gammalab,Chfs= 0,
    muB=1) #excited state 1p1 changed Shfs from 1e6 to 1e7
#changed to yb-174
dijq_D2 = pylcp.hamiltonians.dqij_two_hyperfine_manifolds(3/2, 5/2, 9/2)

E_e_D2 = np.unique(np.diagonal(H_e_D2))
E_g_D2 = np.unique(np.diagonal(H_g_D2))
hamiltonian_D2 = pylcp.hamiltonian()
hamiltonian_D2.add_H_0_block('g', H_g_D2)
hamiltonian_D2.add_H_0_block('e', H_e_D2)
hamiltonian_D2.add_d_q_block('g', 'e', dijq_D2, gamma = gamma, k = k)
hamiltonian_D2.add_mu_q_block('g', mu_q_g_D2)
hamiltonian_D2.add_mu_q_block('e', mu_q_e_D2)



In [ ]:
t0*cts.value('Bohr magneton')/cts.hbar*20e-3

In [ ]:
r = t0*1e-4*np.linspace(-350,750,10000)*(cts.value('Bohr magneton')/cts.hbar)
f_l = np.array([[0,i,0] for i in r ])
eig = {}
mfn = 0
for i in range(0,60):
    eig[f'{i}'] = []
for i in f_l:
    mfn = mfn+1
    k = (np.diagonal(hamiltonian_D2.return_full_H({'g->e':np.array([0,0,0])},i)[18:78,18:78]))
    for j in range(len(k)):
        eig[f'{j}'].append(k[j]*gammalab/1e9)
            
e = 0
n = 0
for i in eig:
    n = n+1
    e = e+1
    if n > 20:
        plt.plot(np.linspace(-350,800,10000),eig[f'{i}'],label = e)
plt.ylim(-6,6)
plt.legend()  
plt.xlabel('B (G)')
plt.ylabel('GHz')      
plt.show()